# Replicate Apollo sample magnetizations (Tarduno, Cottrell & Zhou 2024) on Figshare 24639285

Paper: J. A. Tarduno, R. D. Cottrell, T. Zhou, "Dataset of replicate Apollo sample magnetizations
bearing on impacts and absence of a long-lived lunar dynamo", Scientific Data (2024),
doi:10.1038/s41597-024-03626-0.

Dataset: Figshare item 24639285, version 1, doi:10.6084/m9.figshare.24639285.v1. Version 1 is the
only published version (the API lists no other), and it is the one the paper cites (ref. 26) and the
claims file names, so it is pinned. The cell below prints the MD5 (`computed_md5`) of every file read.
The files are streamed from Figshare's download service into memory; nothing is written to disk.

Claims: `claims/figshare-24639285/10.1038-s41597-024-03626-0.yaml`, written from the version of
record (Europe PMC PMC11271266, full text with Methods, Technical Validation, figure legends and
Tables 1 and 2; the paper has no supplement and no PDF was supplied).

| Claim | Tested here |
|---|---|
| glass-specimen-set (link) | tested |
| glass-success-rate | tested |
| table2-fit-statistics | tested |
| table2-no-fit | tested |
| fig1-linear-components | tested |
| fig2-changing-field | tested |
| glass-directions-not-random | tested |
| glass-af-no-change | tested |
| table1-max-intensity | tested |
| baseline-range | tested |
| example-14053g1-threshold | tested |
| fig3-nrm-consistent | tested |
| fig3-590-inconsistent | tested |
| fig3-alteration-check-inconsistent | tested |
| crystal-14053g1-400c-stable | tested |
| no-alteration-recording-efficiency | tested |
| interpretation-impact-no-dynamo | blocked: an interpretation the claims file says is not tested |

Method details the claims file leaves open are taken from the paper where it gives them, and cited
in `apollo.py`. Where the paper does not settle a detail, every reading is run and shown.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import apollo as A

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

files, glass, crystals = A.load_all()
print(f"Figshare {A.FIGSHARE_ID} version {A.FIGSHARE_VERSION}, doi:{A.DATASET_DOI}")
print(pd.DataFrame([{k: f[k] for k in ("id", "name", "size", "computed_md5")} for f in files]).to_string(index=False))

## Data check

What was loaded: every sheet of every workbook, its columns, the number of rows per specimen, step
kind and replicate, and the ranges and units of moment, declination and inclination. The PDFs
(plots and a PmagPy readme) are listed but not read. The data are static measurements with no time
axis, so there is no clock to align; the check of the description is against the step labels, the
units, the specimen names and the laboratory field.

In [ ]:
for f in files:
    if f["name"].endswith(".xlsx"):
        print(f"{f['name']}: sheets {f['sheets']}")
print()

def summary(name, sheet):
    df = sheet["data"]
    kinds = df.step.map(lambda s: A.step_kind(s)[0])
    reps = df.groupby("step").size()
    return {"specimen": name, "file": sheet["file"], "sample label": ",".join(sorted(df["sample"].unique())),
            "columns": "|".join(map(str, sheet["header"])), "rows": len(df),
            "steps": df.step.nunique(), "AF steps": int((kinds == "af").sum() > 0) * df[kinds == "af"].step.nunique(),
            "thermal T": ",".join(str(int(t)) for t in sorted({A.step_kind(s)[1] for s in df.step if A.step_kind(s)[0] in ("off", "on")})),
            "replicates min-max": f"{reps.min()}-{reps.max()}", "lab field uT": sheet["lab_field_uT"],
            "incomplete rows": sheet["incomplete_rows"]}

gsum = pd.DataFrame([summary(k, v) for k, v in glass.items()])
csum = pd.DataFrame([summary(k, v) for k, v in crystals.items()])
print("Glass, 64455lunar_cryodata.xlsx:")
print(gsum.drop(columns="file").to_string(index=False))
print("\nCrystals, one workbook each:")
print(csum.drop(columns=["file", "lab field uT", "AF steps"]).to_string(index=False))

allrows = pd.concat([v["data"].assign(specimen=k) for d in (glass, crystals) for k, v in d.items()])
print("\nRanges over all measurements: moment %.3g to %.3g emu; declination %.1f to %.1f deg; inclination %.1f to %.1f deg"
      % (allrows.moment.min(), allrows.moment.max(), allrows.dec.min(), allrows.dec.max(), allrows.inc.min(), allrows.inc.max()))
bad = allrows[(allrows.inc.abs() > 90) | (allrows.dec < 0) | (allrows.dec > 360) | (allrows.moment <= 0)]
print("Measurements outside the ranges of a declination (0-360), an inclination (-90 to 90) or a positive moment:")
print(bad[["specimen", "row", "step", "moment", "dec", "inc"]].to_string(index=False))
print("Footnotes of the glass sheets:", sorted({n for v in glass.values() for n in v["notes"]}))
print("Footnotes of the crystal sheets:", sorted({n for v in crystals.values() for n in v["notes"]}))

Repeated measurements of a step should differ by measurement noise. The cell below tests, for every
step of every sheet, whether all its rows are one row multiplied by factors k/100 between 0.90 and
1.10 in moment, declination and inclination alike (`apollo.synthetic_steps`). For four-digit
measurements this cannot happen by chance. It also lists rows repeated exactly within a sheet and
across sheets.

In [ ]:
syn = {k: A.synthetic_steps(v["data"]) for d in (glass, crystals) for k, v in d.items()}
for k, steps in syn.items():
    n = (glass.get(k) or crystals.get(k))["data"].step.nunique()
    if steps:
        print(f"{k}: {len(steps)} of {n} steps are scaled copies of one row: {steps}")
print("Sheets with no such step:", [k for k, s in syn.items() if not s])

def factors(df, step):
    g = df[df.step == step]
    out = {}
    for c in ("moment", "dec", "inc"):
        v = g[c].abs().to_numpy()
        for v0 in v:
            for f0 in np.arange(90, 111) / 100:
                r = v / (v0 / f0)
                if np.all(np.abs(r * 100 - np.round(r * 100)) < 1e-6 * r * 100):
                    out[c] = np.round(r, 2).tolist()
                    break
            if c in out:
                break
    return out

for k in ("g2", "ss8", "ss39", "ss48"):
    print(f"{k} NRM rows, factors relative to the base row:", factors(glass[k]["data"], "0"))
print(glass["ss8"]["data"].query("step == '0'")[["row", "moment", "dec", "inc"]].to_string(index=False))

dups = []
for k, v in {**glass, **crystals}.items():
    d = v["data"]
    n = d.duplicated(["step", "moment", "dec", "inc"]).sum()
    if n:
        dups.append((k, int(n), k in ("g2", "ss8", "ss39", "ss48")))
print("Rows repeated exactly within a sheet (specimen, count, synthetic sheet):", dups)
key = allrows.set_index(["moment", "dec", "inc"])
cross = allrows.groupby(["moment", "dec", "inc"]).specimen.nunique()
print("Rows found in more than one sheet:")
print(allrows.set_index(["moment", "dec", "inc"]).loc[cross[cross > 1].index][["specimen", "row", "step"]])

Checks of the description against the data:

- Columns: every sheet has sample, step, moment (emu), declination and inclination, as the Data
  Records say, except ss48, which has an extra column `CSD` (empty or 0) before the moment. The
  moment column is found by name.
- Specimen names: the glass sheet `ss11` is named `s11` in Table 2; the sheet `g2` labels its
  rows `64455g2`. Twelve crystal workbooks match the twelve names of Table 1.
- Step labels follow the description (`#.0`, `#.1`, `#.2` for the glass; `#.3`, `#.4` for the
  crystals); NRM appears as both `0` and `0.0` (ss27, ss38).
- Laboratory field: the glass sheets give 18 uT (10 sheets), 15 uT (11) or 14.4 uT (g2); the paper
  gives 18 uT only for Figure 1, whose specimens (ss19, ss43) are 18 uT sheets. The crystal sheets
  give 20 and 40 uT for `#.1` and `#.4`, as the Methods say.
- Replicates: each glass step was measured 2 to 19 times, each crystal NRM 1 to 3 times, each 590
  zero-field step 3 or 4 times, and each crystal field-on step once. The paper does not say how
  repeated measurements enter Table 2; both the mean vector and the first measurement are run below.
- Four glass sheets (g2, ss8, ss39, ss48) do not hold repeated measurements: every step's rows are
  one row times factors near 1. The analysis keeps them, as the deposit presents them, and the
  results that use them say so; their mean vector is close to the base row.
- Values out of range: one inclination of -136 deg (ss19, 10 mT, almost certainly -13.6), one
  declination of 361.7 deg (ss8 NRM, a scaled copy), and an ss18 row at 250 C with no values. The
  first two are used as they are (converted to vectors, a declination of 361.7 is 1.7 deg); the
  ss19 value is in an AF step no analysis below uses; the empty row is dropped.

## glass-specimen-set (link)

> The deposit holds the measurements of the 22 Apollo 64455 glass subsamples that failed the
> paleointensity selection criteria, named as in Table 2, and replicate measurements for the twelve
> single crystals of Table 1.

Criterion: specimens matched by the names of Tables 1 and 2 (s11 may be ss11); each has steps,
moments, declinations and inclinations; which glass specimens have AF steps is compared with the
reading of the Table 2 marks (12 without, 10 with).

In [ ]:
TABLE2 = ["ss19", "ss43", "ss22", "ss41", "ss6", "ss9", "ss10", "ss30", "ss34", "g2", "ss8", "s11", "ss18",
          "ss24", "ss27", "ss29", "ss36", "ss38", "ss39", "ss45", "ss46", "ss48"]
# Table 2 marks: the second mark (triangle) read as AF pre-treatment (claims file, opening comment)
TABLE2_AF_MARK = ["ss19", "ss43", "ss22", "ss41", "ss34", "ss27", "ss38", "ss45", "ss46", "ss48"]
TABLE1 = ["12021g2", "12021g3", "12040g1", "12040g2", "12040g3", "12053g1", "12053g2", "14053g1", "14053g2",
          "14053g3", "71055g1", "71055g3"]
SHEET = {n: ("ss11" if n == "s11" else n) for n in TABLE2}

def complete(sheet):
    d = sheet["data"]
    return len(d) > 0 and d[["moment", "dec", "inc"]].notna().all().all() and d.step.nunique() > 1

found_glass = [n for n in TABLE2 if SHEET[n] in glass and complete(glass[SHEET[n]])]
found_crystals = [n for n in TABLE1 if n in crystals and complete(crystals[n])]
extra = sorted(set(glass) - set(SHEET.values())) + sorted(set(crystals) - set(TABLE1))
has_af = sorted(n for n in TABLE2 if (glass[SHEET[n]]["data"].step.str.endswith("mT")).any())
print(f"Glass specimens of Table 2 found with complete measurements: {len(found_glass)} of {len(TABLE2)}")
print(f"Crystals of Table 1 found with replicate measurements: {len(found_crystals)} of {len(TABLE1)}")
print("Sheets or files not named in the tables:", extra)
print(f"Glass specimens with AF steps: {len(has_af)} {has_af}; without: {len(TABLE2) - len(has_af)}")
print("Same set as the Table 2 marks:", set(has_af) == set(TABLE2_AF_MARK))
print("Crystals with more than one NRM, 590.0 and 590.3 measurement:",
      {n: [int((crystals[n]['data'].step == s).sum()) for s in ('0', '590', '590.3')] for n in TABLE1})

**Result: reproduced.** All 22 glass specimens of Table 2 (s11 as sheet ss11) and all 12 crystals
of Table 1 are in the deposit with steps, moments, declinations and inclinations, and no other
specimen is. The 10 glass specimens with AF steps are exactly those carrying the second mark in
Table 2. Two crystals (71055g1, 71055g3) have a single NRM measurement. Four glass sheets hold
scaled copies of one measurement in place of replicates (Data check).

## glass-success-rate

> Of the Apollo 64455 glass subsamples given thermal paleointensity experiments, 12% passed the
> selection criteria (3 of 25, 13 run without and 12 with AF pre-treatment).

Criterion: rate = 3 / (3 + failed specimens found here); the AF split is counted from AF steps in
the deposit plus the Methods' successful ones (one without, two with AF). Holds when 22 failed
specimens are found and 10 have AF steps.

In [ ]:
n_failed = len(found_glass)
n_af = len(has_af)
rate = 100 * 3 / (3 + n_failed)
print(f"Failed specimens in the deposit: {n_failed}; with AF steps {n_af}, without {n_failed - n_af}")
print(f"Success rate 3 / (3 + {n_failed}) = {rate:.1f}% (expected 12 +- 1.2)")
print(f"Without AF: {n_failed - n_af} + 1 = {n_failed - n_af + 1} (expected 13); with AF: {n_af} + 2 = {n_af + 2} (expected 12)")

**Result: reproduced.** 22 failed specimens give 3/25 = 12.0%, and the AF steps split them 12 and
10, which with the Methods' successful specimens gives 13 without and 12 with AF pre-treatment.

## table2-fit-statistics

> For the nine failed glass specimens with a line fit (ss19, ss43, ss22, ss41, ss6, ss9, ss10, ss30,
> ss34), the number of points, temperature range, MAD, R2 and pTRM check differences are as in
> Table 2, which lists the criteria each failed.

Criterion: per specimen, the paired steps in the Table 2 range, NRM against TRM gained, R2 of the
least-squares line, MAD under readings (a) free, (b) anchored, (c) free on the three-point running
mean, and pTRM check percentages under readings (a) over the original pTRM, (b) over the length of the
fitted segment, with the check's pTRM referred to (i) the field-off step just before it or (ii) the
field-off step at its temperature. A value matches when one reading puts it within tolerance.
Reproduced when n matches for all nine and at least 80% of the MAD, R2 and pTRM values match;
partially for 50 to 80%.

The paper does not say how the repeated measurements of a step are combined. Both the mean vector
and the first measurement are run; the mean is adopted as an assumption (see the end of this
section).

In [ ]:
T2 = {  # Table 2: n, T range, MAD, R2, pTRM checks {T: %}, failure criteria
    "ss19": (4, 175, 450, 30.1, 0.94, {250: 30, 450: 9}, "pTRM, MAD, n"),
    "ss43": (3, 250, 400, 10.0, 0.90, {250: 85}, "pTRM, n"),
    "ss22": (4, 250, 500, 35.4, 0.38, {250: 22, 450: 13}, "pTRM, MAD, R2"),
    "ss41": (3, 500, 580, 24.0, 0.38, {250: 4, 400: 24}, "pTRM, MAD, R2, n"),
    "ss6": (3, 175, 360, 21.0, 0.91, {}, "MAD, R2, n, no pTRM"),
    "ss9": (4, 250, 550, 37.6, 0.74, {250: 21}, "pTRM, MAD, R2"),
    "ss10": (3, 360, 550, 42.9, 0.15, {250: 83}, "pTRM, MAD, R2, n"),
    "ss30": (4, 175, 500, 12.5, 0.59, {450: 10}, "R2"),
    "ss34": (5, 100, 450, 12.1, 0.06, {250: 9}, "R2"),
}
TOL_MAD = {"ss19": 3.0, "ss43": 1.0, "ss22": 3.5, "ss41": 2.4, "ss6": 2.1, "ss9": 3.8, "ss10": 4.3, "ss30": 1.25, "ss34": 1.2}

def fit_stats(how):
    rows, checks = [], []
    for s, (n, t0, t1, mad, rr, pt, fail) in T2.items():
        df = glass[s]["data"]
        a = A.in_range(A.arai(df, how), t0, t1)
        m = A.mads(df, t0, t1, how)
        r = A.r2(a.trm, a.nrm)
        rows.append({"specimen": s, "how": how, "n": len(a), "n_T2": n, "steps": ",".join(str(int(t)) for t in a.t),
                     "R2": r, "R2_T2": rr, "R2_ok": abs(r - rr) <= 0.1 * rr + 1e-9,
                     **{f"MAD_{k}": v for k, v in m.items()}, "MAD_T2": mad,
                     "MAD_ok_any": any(abs(v - mad) <= TOL_MAD[s] + 1e-9 for v in m.values()),
                     "MAD_ok_a": abs(m["a_free"] - mad) <= TOL_MAD[s] + 1e-9})
        p = A.ptrm_checks(df, how, (t0, t1))
        for T, pct in pt.items():
            q = p[p.t == T]
            vals = {f"{col[0]}_{ref.split('_')[0]}": float(q[q.ref == ref][col].iloc[0])
                    for col in ("a_pct", "b_pct") for ref in ("i_prev", "ii_same")}
            checks.append({"specimen": s, "how": how, "T": T, "printed": pct, **{k: round(v, 1) for k, v in vals.items()},
                           **{f"ok_{k}": abs(v / pct - 1) <= 0.1 + 1e-9 for k, v in vals.items()},
                           "prev_off_T": int(q.prev_off_t.iloc[0])})
    rows, checks = pd.DataFrame(rows), pd.DataFrame(checks)
    checks["ok_any"] = checks[[c for c in checks if c.startswith("ok_")]].any(axis=1)
    return rows, checks

fits, chks = {}, {}
for how in ("mean", "first"):
    fits[how], chks[how] = fit_stats(how)
    print(f"--- replicates combined as: {how}")
    print(fits[how].round(2).to_string(index=False))
    print(chks[how].to_string(index=False))

In [ ]:
def score(how):
    f, c = fits[how], chks[how]
    n_ok = (f.n == f.n_T2).all()
    mad_ok, r2_ok, pt_ok = int(f.MAD_ok_any.sum()), int(f.R2_ok.sum()), int(c.ok_any.sum())
    total = len(f) * 2 + len(c)
    return n_ok, mad_ok, r2_ok, pt_ok, total

for how in ("mean", "first"):
    n_ok, mad_ok, r2_ok, pt_ok, total = score(how)
    print(f"{how}: n matches for all nine {n_ok}; MAD {mad_ok}/9, R2 {r2_ok}/9, pTRM {pt_ok}/{len(chks[how])}; "
          f"together {mad_ok + r2_ok + pt_ok}/{total} = {100 * (mad_ok + r2_ok + pt_ok) / total:.0f}%")

# How often a single reading matches, so that the any-reading count can be judged
f, c = fits["mean"], chks["mean"]
for k in ("a_free", "b_anchored", "c_smoothed"):
    ok = sum(abs(f[f"MAD_{k}"] - f.MAD_T2) <= f.specimen.map(TOL_MAD) + 1e-9)
    print(f"mean, MAD reading {k} alone: {ok}/9 within tolerance; median |difference| "
          f"{np.median(np.abs(f[f'MAD_{k}'] - f.MAD_T2)):.1f} deg")
for k in ("a_i", "a_ii", "b_i", "b_ii"):
    print(f"mean, pTRM reading {k} alone: {int(c[f'ok_{k}'].sum())}/{len(c)} within 10% of the printed value")
print("Values outside tolerance under every reading (mean):")
print(f.loc[~f.MAD_ok_any, ["specimen", "MAD_a_free", "MAD_b_anchored", "MAD_c_smoothed", "MAD_T2"]].round(1).to_string(index=False))
print(f.loc[~f.R2_ok, ["specimen", "R2", "R2_T2"]].round(3).to_string(index=False))
print(c.loc[~c.ok_any, ["specimen", "T", "printed", "a_i", "a_ii", "b_i", "b_ii"]].to_string(index=False))

In [ ]:
# Failure criteria against the stated thresholds (n >= 4, pTRM within 15%, R2 about 0.9); reported, not judged
f, c = fits["mean"], chks["mean"]
rep = []
for s, (n, t0, t1, mad, rr, pt, fail) in T2.items():
    fr = f[f.specimen == s].iloc[0]
    cs = c[c.specimen == s]
    rep.append({"specimen": s, "Table 2 failure": fail, "n<4": fr.n < 4, "R2<0.9": round(fr.R2, 2) < 0.9,
                "pTRM>15% (reading a_i)": bool((cs.a_i > 15).any()) if len(cs) else None, "MAD (mean, a)": round(fr.MAD_a_free, 1)})
print(pd.DataFrame(rep).to_string(index=False))
print("ss6 250.1 holds 19 rows in two groups of directions:")
d = glass["ss6"]["data"].query("step == '250.1'")
print(d.assign(group=np.where(d.inc > 30, "inc>30", "inc<30")).groupby("group")[["moment", "dec", "inc"]].agg(["count", "mean"]).round(10))

**Result: partially reproduced, resting on the assumption that repeated measurements are averaged.**
With the mean vector of each step's repeated measurements, n matches Table 2 for all nine specimens,
and 23 of the 29 MAD, R2 and pTRM values match under one of the named readings (79%, just under the
80% the criterion asks for): MAD 8 of 9, R2 6 of 9, pTRM checks 9 of 11. MAD reading (a), a free
line, alone gives the 8 matching MADs, 6 of them within 0.3 deg of the printed value. ss10 differs
in MAD (33.3 against 42.9 deg) and R2 (0.23 against 0.15); ss9 differs in R2 (0.89 against 0.74);
and ss34's R2 is 0.03 against 0.06, outside a tolerance of 0.006. The matching pTRM checks need
different readings for different specimens (no single reading matches more than 5 of 11), so no
single pTRM definition is confirmed. With the first measurement of each step, 11 of 29 values match
(38%), which would make the claim not reproduced; the mean is adopted because it gives the printed
MADs to the printed digit for six specimens. ss6's 250.1 step mixes two groups of directions (19
rows), which lowers its R2 under the mean.

## table2-no-fit

> The other thirteen failed glass specimens (g2, ss8, s11, ss18, ss24, ss27, ss29, ss36, ss38, ss39,
> ss45, ss46, ss48) gave no fit, most because they did not acquire TRM in a regular fashion and some
> because their directions trend out of the origin.

Criterion: over each Table 2 range, TRM gained per step; regular acquisition read as (a) TRM gained
increasing at every step, or (b) no step falling by more than 15%. Reproduced when, under one reading,
at least 9 of the 11 specimens marked "no regular TRM acquisition" are irregular, and ss36 and ss39
have field-off vectors whose distance from the origin grows over the last steps. The pTRM checks are
reported.

In [ ]:
NOFIT = {"g2": (4, 100, 400, {}), "ss8": (5, 100, 450, {}), "s11": (6, 100, 550, {250: 22}), "ss18": (5, 100, 450, {250: 66}),
         "ss24": (7, 100, 550, {250: 6, 450: 42}), "ss27": (6, 100, 550, {305: 1, 400: 54}), "ss29": (6, 100, 550, {450: 25}),
         "ss36": (4, 100, 450, {450: 25}), "ss38": (4, 100, 360, {250: 9}), "ss39": (2, 100, 210, {}),
         "ss45": (4, 100, 360, {250: 120}), "ss46": (3, 100, 250, {}), "ss48": (2, 100, 175, {})}
IRREGULAR_LABEL = [s for s in NOFIT if s not in ("ss36", "ss39")]
SYNTH = ["g2", "ss8", "ss39", "ss48"]

def regularity(how):
    rows = []
    for s, (n, t0, t1, pt) in NOFIT.items():
        a = A.in_range(A.arai(glass[SHEET[s]]["data"], how), t0, t1)
        trm = a.trm.to_numpy()
        ratio = trm[1:] / trm[:-1]
        rows.append({"specimen": s, "how": how, "n": len(a), "n_T2": n, "T": ",".join(str(int(t)) for t in a.t),
                     "TRM gained (1e-9 emu)": " ".join(f"{x * 1e9:.2f}" for x in trm),
                     "irregular_a": bool((ratio <= 1).any()), "irregular_b": bool((ratio < 0.85).any()),
                     "largest fall %": round(100 * (1 - ratio.min()), 1) if len(ratio) else None,
                     "|off| (1e-9 emu)": " ".join(f"{x * 1e9:.2f}" for x in a.nrm),
                     "synthetic sheet": s in SYNTH})
    return pd.DataFrame(rows)

for how in ("mean", "first"):
    r = regularity(how)
    print(f"--- {how}")
    print(r.to_string(index=False))
    lab = r[r.specimen.isin(IRREGULAR_LABEL)]
    print(f"{how}: irregular under (a) {int(lab.irregular_a.sum())}/11, under (b) {int(lab.irregular_b.sum())}/11; "
          f"without the four synthetic sheets (a) {int(lab[~lab['synthetic sheet']].irregular_a.sum())}/{int((~lab['synthetic sheet']).sum())}, "
          f"(b) {int(lab[~lab['synthetic sheet']].irregular_b.sum())}/{int((~lab['synthetic sheet']).sum())}")
    for s in ("ss36", "ss39"):
        a = A.in_range(A.arai(glass[s]["data"], how), *NOFIT[s][1:3])
        print(f"  {s}: distance of the field-off vector from the origin over the range "
              f"{[f'{x * 1e9:.2f}' for x in a.nrm]} (1e-9 emu); grows over the last step: {bool(a.nrm.iloc[-1] > a.nrm.iloc[-2])}")

In [ ]:
# A null for the irregularity counts: how often 'irregular' arises from measurement scatter alone.
# For each step of the non-synthetic specimens, the spread of TRM gained across replicates
# (the field-on and field-off rows paired in order) is compared with the step-to-step falls.
rng = np.random.default_rng(2101)
rows = []
for s in IRREGULAR_LABEL:
    if s in SYNTH:
        continue
    df = glass[SHEET[s]]["data"]
    a = A.in_range(A.arai(df, "mean"), *NOFIT[s][1:3])
    sd = []
    for t in a.t:
        on = df[df.step == f"{t:g}.1"]; off = df[df.step.isin([f"{t:g}", f"{t:g}.0"])]
        k = min(len(on), len(off))
        x = np.linalg.norm(A.to_xyz(on.moment[:k], on.dec[:k], on.inc[:k]) - A.to_xyz(off.moment[:k], off.dec[:k], off.inc[:k]), axis=1)
        sd.append(x.std(ddof=1) / np.sqrt(k))
    rows.append({"specimen": s, "median SE of mean TRM gained / mean TRM gained": round(float(np.median(np.array(sd) / a.trm)), 3)})
print(pd.DataFrame(rows).to_string(index=False))

print("pTRM checks of the no-fit specimens (mean, readings a and b over the Table 2 range), against Table 2:")
out = []
for s, (n, t0, t1, pt) in NOFIT.items():
    p = A.ptrm_checks(glass[SHEET[s]]["data"], "mean", (t0, t1))
    for T, pct in pt.items():
        q = p[p.t == T]
        if len(q) == 0:
            out.append({"specimen": s, "T": T, "printed": pct, "note": "no check at this T in the sheet"})
            continue
        vals = {f"{c[0]}_{r.split('_')[0]}": round(float(q[q.ref == r][c].iloc[0]), 1) for c in ("a_pct", "b_pct") for r in ("i_prev", "ii_same")}
        out.append({"specimen": s, "T": T, "printed": pct, **vals,
                    "any within 10%": any(abs(v / pct - 1) <= 0.1 for v in vals.values())})
out = pd.DataFrame(out)
print(out.to_string(index=False))
print(f"pTRM checks matching under some reading: {int(out['any within 10%'].fillna(False).sum())}/{len(out)}")

**Result: reproduced.** Under reading (a), all 11 specimens marked "no regular TRM acquisition"
acquire TRM irregularly (9 of 11 under reading b, and 8 of the 8 not from synthetic sheets under
reading a); ss36 and ss39 have field-off vectors that move away
from the origin over the last step of their ranges. Three of the 11 (g2, ss8, ss48), and ss39, are
synthetic sheets. Reading (a) is weak: any fall in TRM gained counts, and the median standard error of
the mean TRM gained is printed above as a scale for the smaller falls. The pTRM checks of these
specimens are reported only: 6 of 10 match under some reading, and ss36 has no check at 450 C in
the sheet although Table 2 prints one; ss27 has 5 paired steps from 100 to 550 C where Table 2 gives 6.

## fig1-linear-components

> Glass subsamples ss43 and ss19, which failed the reliability criteria, show hints of linear
> directional components on orthogonal vector plots, from which nominal paleointensities can be
> calculated with the 18 uT laboratory field.

Criterion: over the Table 2 ranges, MAD and R2 as in table2-fit-statistics, and the nominal
paleointensity as the absolute slope times 18 uT (Figure 1 legend). Holds when both MADs and R2
values match Table 2 under one reading.

In [ ]:
for how in ("mean", "first"):
    f = fits[how].set_index("specimen")
    for s in ("ss43", "ss19"):
        n, t0, t1 = T2[s][:3]
        a = A.in_range(A.arai(glass[s]["data"], how), t0, t1)
        b = A.slope(a.trm, a.nrm)
        print(f"{how} {s}: MAD a/b/c {f.loc[s, 'MAD_a_free']:.1f}/{f.loc[s, 'MAD_b_anchored']:.1f}/{f.loc[s, 'MAD_c_smoothed']:.1f} "
              f"(Table 2 {T2[s][3]}), R2 {f.loc[s, 'R2']:.3f} (Table 2 {T2[s][4]}), slope {b:.3f}, "
              f"lab field in sheet {glass[s]['lab_field_uT']} uT, nominal paleointensity |slope| x 18 = {abs(b) * 18:.1f} uT")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
lim = []
for ax, s in zip(axes, ("ss43", "ss19", "ss41", "ss22")):
    a = A.arai(glass[s]["data"], "mean")
    t0, t1 = T2[s][1:3]
    sel = (a.t >= t0) & (a.t <= t1)
    ax.plot(a.trm * 1e9, a.nrm * 1e9, "o-", color="0.6")
    ax.plot(a.trm[sel] * 1e9, a.nrm[sel] * 1e9, "o", color="k")
    for t, x, y in zip(a.t, a.trm * 1e9, a.nrm * 1e9):
        ax.annotate(f"{t:g}", (x, y), fontsize=7)
    ax.set_title(f"{s} (mean of replicates)")
    ax.set_xlabel("TRM gained (1e-9 emu)")
    ax.set_ylabel("NRM remaining (1e-9 emu)")
    lim.append((s, a.trm.min() * 1e9, a.trm.max() * 1e9, a.nrm.min() * 1e9, a.nrm.max() * 1e9))
plt.tight_layout()
plt.show()
print("Plotted ranges (x min, x max, y min, y max, 1e-9 emu); all points are inside the axes:")
for r in lim:
    print(r[0], [round(float(v), 2) for v in r[1:]])

**Result: reproduced, resting on the averaging assumption.** With averaged replicates and a free line,
ss43 has MAD 10.3 deg and R2 0.90 (Table 2: 10.0 and 0.90), and ss19 MAD 30.1 deg and R2 0.96 (30.1
and 0.94). The nominal paleointensities are 9.6 uT for ss43 and 23.0 uT for ss19 (8.5 and 23.7 uT with
first measurements); the figure's values were not available to compare.

## fig2-changing-field

> Glass subsamples ss41 and ss22 show field-off directions that change over a temperature interval,
> consistent with a changing ambient field during magnetization, and curved NRM lost against TRM
> gained plots.

Criterion: Table 2 ranges (ss41 500 to 580 C, ss22 250 to 500 C). Readings (a) the angle between the
first and last field-off direction of the range exceeds the MAD of a line fit over the range, (b) a
great-circle fit has a smaller MAD than a line fit. Reproduced when both specimens change under one
reading and R2 is within tolerance of 0.38.

In [ ]:
for how in ("mean", "first"):
    for s in ("ss41", "ss22"):
        t0, t1 = T2[s][1:3]
        seq = A.demag_sequence(glass[s]["data"], how)
        r = seq[(seq.kind == "off") & (seq.t >= t0) & (seq.t <= t1)]
        X = r[["x", "y", "z"]].to_numpy()
        ang = A.angle(X[0], X[-1])
        line_free, line_anch, plane = A.mad_line(X), A.mad_line(X, anchored=True), A.mad_plane(X)
        dirs = [tuple(round(v, 1) for v in A.to_dir(x)[:2]) for x in X]
        r2v = fits[how].set_index("specimen").loc[s, "R2"]
        print(f"{how} {s}: steps {list(r.t.astype(int))}, directions (dec, inc) {dirs}")
        print(f"   first-last angle {ang:.1f} deg; line MAD free {line_free:.1f}, anchored {line_anch:.1f}; "
              f"great-circle MAD {plane:.1f}; (a) {ang > line_free}, (b) {plane < line_free}; R2 {r2v:.3f} (0.38 +- 0.038)")

**Result: reproduced, resting on the averaging assumption.** For both specimens the field-off
direction turns by more than the MAD of a line over the range, and a great circle fits better than a
line (numbers above); R2 is 0.38 for both with averaged replicates. With first measurements ss41's R2
is 0.16.

## glass-directions-not-random

> The field-off directions of the failed glass specimens, as seen on orthogonal vector plots, are not
> random, implying the presence of an ambient field.

Criterion (a standard choice, the paper names no test): per specimen, Watson's test of randomness on
the field-off directions of its thermal steps (p < 0.05). Reproduced when more than half of the 22
reject randomness, partially for a quarter to a half. Also given for the steps within each Table 2
range.

In [ ]:
RANGES = {**{s: v[1:3] for s, v in T2.items()}, **{s: v[1:3] for s, v in NOFIT.items()}}
rows = []
for i, n in enumerate(TABLE2):
    s = SHEET[n]
    for how in ("mean", "first"):
        st = A.step_table(glass[s]["data"], how)
        off = st[st.kind == "off"]
        f = A.fisher(off[["x", "y", "z"]].to_numpy())
        p = A.watson_p(f["n"], f["R"], seed=3000 + i)
        t0, t1 = RANGES[n]
        o2 = off[(off.t >= t0) & (off.t <= t1)]
        f2 = A.fisher(o2[["x", "y", "z"]].to_numpy())
        p2 = A.watson_p(f2["n"], f2["R"], seed=3100 + i)
        rows.append({"specimen": n, "how": how, "n": f["n"], "R": round(f["R"], 2), "k": round(f["k"], 1), "p": p,
                     "n_range": f2["n"], "R_range": round(f2["R"], 2), "p_range": p2, "synthetic sheet": n in SYNTH})
w = pd.DataFrame(rows)
print(w.to_string(index=False))
for how in ("mean", "first"):
    x = w[w.how == how]
    print(f"{how}: all thermal field-off steps reject randomness {int((x.p < 0.05).sum())}/22; "
          f"within Table 2 ranges {int((x.p_range < 0.05).sum())}/22 "
          f"(ranges of 2 steps: {int((x.n_range <= 2).sum())}); without synthetic sheets {int((x[~x['synthetic sheet']].p < 0.05).sum())}/18")

**Result: partially reproduced.** Watson's test rejects randomness of the thermal field-off
directions in 8 of 22 specimens with averaged replicates (36%, within the quarter to half band), 9
of 22 with first measurements, and 8 of 22 within the Table 2 ranges. The remaining specimens give directions that the
test cannot tell from random at these numbers of steps (2 to 9).

## glass-af-no-change

> For nearly all glass samples given AF pre-treatment, AF cleaning did not significantly change the
> NRM direction or magnitude.

Criterion: per specimen with AF steps, the angle between the NRM and the direction after the last AF
step, and the ratio of the moments. Readings (a) angle under 15 deg and moment change under 15%; (b)
angle under the Table 2 MAD where printed, and moment change under 15%. Reproduced when, under one
reading, all but at most two show no significant change.

In [ ]:
MADT2 = {s: v[3] for s, v in T2.items()}
rows = []
for s in has_af:
    for how in ("mean", "first"):
        st = A.step_table(glass[s]["data"], how)
        nrm = st[st.kind == "nrm"].iloc[0]
        last = st[st.kind == "af"].iloc[-1]
        ang = A.angle(A.vec(nrm), A.vec(last))
        ratio = last.moment / nrm.moment
        lim = MADT2.get(s, 15.0)
        rows.append({"specimen": s, "how": how, "last AF": last.step, "angle": round(ang, 1), "moment ratio": round(ratio, 3),
                     "ok_a": ang < 15 and abs(ratio - 1) < 0.15, "MAD limit (b)": lim,
                     "ok_b": ang < lim and abs(ratio - 1) < 0.15, "synthetic sheet": s in SYNTH})
af = pd.DataFrame(rows)
print(af.to_string(index=False))
for how in ("mean", "first"):
    x = af[af.how == how]
    print(f"{how}: no significant change under (a) {int(x.ok_a.sum())}/{len(x)}, under (b) {int(x.ok_b.sum())}/{len(x)}")
print("ss38 NRM rows ('0' and '0.0'):")
print(glass["ss38"]["data"].query("step in ['0', '0.0']")[["row", "step", "moment", "dec", "inc"]].to_string(index=False))

**Result: reproduced.** AF cleaning turns the NRM by at most 2.8 deg and changes its moment by at
most 1.1% in 8 of the 10 specimens; ss38 keeps its direction but loses half its moment (ratio 0.49 at
20 mT) and ss48, a synthetic sheet, turns by 17.5 deg and loses a third. Two exceptions are allowed.

## table1-max-intensity

> For each of the twelve single crystals, the maximum field that cannot be excluded, given the
> magnetometer threshold of 8e-11 emu and the moment acquired at 590 C in a 20 uT field, is as in
> Table 1, between 0.20 and 0.51 uT.

Criterion: max intensity = 8e-11 emu / M590 x 20 uT. M590 under readings (a) |590.1 - 590.0|, (b)
|590.1|, with (i) the first 590.0 replicate or (ii) their mean. Reproduced when, under one reading,
all 24 values are within tolerance; partially when at least 18 are.

In [ ]:
T1 = {"12021g2": (7.011e-9, 0.23), "12021g3": (6.292e-9, 0.26), "12040g1": (8.002e-9, 0.20), "12040g2": (6.021e-9, 0.27),
      "12040g3": (6.292e-9, 0.26), "12053g1": (5.283e-9, 0.31), "12053g2": (7.982e-9, 0.20), "14053g1": (3.203e-9, 0.51),
      "14053g2": (5.294e-9, 0.31), "14053g3": (4.523e-9, 0.36), "71055g1": (3.883e-9, 0.42), "71055g3": (4.202e-9, 0.39)}

def grp(name, step):
    d = crystals[name]["data"]
    d = d[d.step == step]
    return A.to_xyz(d.moment, d.dec, d.inc)

def m590(name, reading):
    on = grp(name, "590.1")[0]
    off = grp(name, "590")
    off = off[0] if reading.endswith("_i") else off.mean(axis=0)
    return float(np.linalg.norm(on - off)) if reading.startswith("a") else float(np.linalg.norm(on))

rows = []
for name, (m_t1, i_t1) in T1.items():
    r = {"crystal": name, "M590 T1": m_t1, "max T1": i_t1}
    for rd in ("a_i", "a_ii", "b_i", "b_ii"):
        m = m590(name, rd)
        r[f"M_{rd}"] = m
        r[f"max_{rd}"] = round(A.max_intensity(m), 3)
        r[f"ok_{rd}"] = int(abs(m - m_t1) <= 0.1 * m_t1) + int(abs(A.max_intensity(m) - i_t1) <= 0.1 * i_t1 + 1e-9)
    r["T1 formula on own M590"] = round(A.max_intensity(m_t1), 3)
    rows.append(r)
t1 = pd.DataFrame(rows)
print(t1.to_string(index=False))
for rd in ("a_i", "a_ii", "b_i", "b_ii"):
    print(f"reading {rd}: {int(t1[f'ok_{rd}'].sum())}/24 values within tolerance")
print("Table 1 max intensity equal to the formula on its own M590, to two places:",
      int((t1["T1 formula on own M590"].round(2) == t1["max T1"]).sum()), "/ 12")
import math
print("Table 1 max intensity equal to the formula on its own M590, rounded up to two places:",
      int(sum(math.ceil(round(A.max_intensity(m) * 100, 6)) / 100 == i for m, i in zip(t1["M590 T1"], t1["max T1"]))), "/ 12")
print("Formula on Table 1's own M590, unrounded:", {n: round(A.max_intensity(m), 4) for n, m in zip(t1.crystal, t1["M590 T1"])})
print("Exact equality of M590 (b) with Table 1 (to 4 significant digits):",
      int(sum(float(f"{m:.4g}") == mt for m, mt in zip(t1.M_b_i, t1["M590 T1"]))), "/ 12")
print("12021g3 and 12040g3 590.1 rows:")
print(pd.concat([crystals[n]["data"].query("step == '590.1'") for n in ("12021g3", "12040g3")]).to_string(index=False))

**Result: reproduced.** Under reading (b), M590 the magnitude of the single 590.1 measurement, all 24
values are within tolerance: the 12 M590 values equal Table 1 to its four digits and the max
intensities run from 0.200 to 0.500 uT. Reading (a) with averaged 590.0 replicates also puts all 24
within tolerance (23 with the first replicate). Table 1's max intensities equal the formula on its
own M590 column, rounded to two places, for only 4 of 12 crystals; rounded up, for 10 of 12, the
exceptions being 14053g1 (0.4995 against 0.51) and 12053g2 (0.2005 against 0.20). The equal M590 of 12021g3 and 12040g3
is in the data: both 590.1 rows read 6.292e-09 emu, with different directions.

## baseline-range

> The maximum fields that the single crystal data cannot exclude define a baseline of about 0.2 to
> 0.5 uT below which fields cannot be evaluated with these data.

Criterion: the lowest and highest of the twelve values of table1-max-intensity, under the reading
that matches there, lie in 0.18 to 0.22 and 0.45 to 0.55 uT.

In [ ]:
vals = t1.max_b_i
print(f"Reading (b): lowest {vals.min():.3f} uT ({t1.crystal[vals.idxmin()]}), highest {vals.max():.3f} uT ({t1.crystal[vals.idxmax()]})")
print(f"Within 0.18-0.22: {0.18 <= vals.min() <= 0.22}; within 0.45-0.55: {0.45 <= vals.max() <= 0.55}")
for rd in ("a_i", "a_ii"):
    print(f"Reading {rd} for comparison: {t1[f'max_{rd}'].min():.3f} to {t1[f'max_{rd}'].max():.3f} uT")

**Result: reproduced.** The twelve values run from 0.200 to 0.500 uT.

## example-14053g1-threshold

> For crystal 14053g1, the threshold field detection limit computed from the magnetometer sensitivity
> (8e-14 A m2) and a TRM at 590 C of 8e-12 A m2 in a 20 uT field is 0.2 uT.

Criterion: readings (a) the example as written, for 14053g1; (b) a misprint of the crystal name, the
numbers belonging to 12040g1. TRM as in table1-max-intensity (reading b matched there). Reproduced as
written when (a) matches; when only (b) matches, reported as matching the misprint reading.

In [ ]:
for name in ("14053g1", "12040g1"):
    m_emu = m590(name, "b_i")
    trm_am2 = m_emu * 1e-3   # 1 emu = 1e-3 A m2
    thr = 8e-14 / trm_am2 * 20
    print(f"{name}: TRM at 590 C {trm_am2:.3e} A m2 (expected 8.0e-12 +- 8e-13: {abs(trm_am2 - 8e-12) <= 8e-13}); "
          f"threshold field {thr:.3f} uT (expected 0.2 +- 0.02: {abs(thr - 0.2) <= 0.02})")

**Result: reproduced under the misprint reading (b).** For 14053g1 the data give a TRM of 3.2e-12 A m2
and 0.50 uT, not the example's numbers; for 12040g1 they give 8.0e-12 A m2 and 0.200 uT, the example's
numbers. The example's crystal name is a misprint for 12040g1.

## fig3-nrm-consistent, fig3-590-inconsistent, fig3-alteration-check-inconsistent

> For most crystals the replicate NRM measurements have a consistent direction, as shown on stereonets
> for 12040g1 and 14053g2. After demagnetization at 590 C in zero field, the replicate directions are
> inconsistent, consistent with a null magnetization state. The replicate directions after the
> zero-field alteration check at 590 C are also inconsistent.

Criteria: per crystal, Fisher mean, k and alpha95 of the replicate directions of each step. NRM
consistent under (a) k of the NRM replicates above k of the 590.0 replicates, or (b) Watson's test
rejects randomness of the NRM replicates (p < 0.05). 590.0 (and 590.3) inconsistent under (a) k below
the NRM's k, or (b) Watson's test does not reject randomness. Each reproduced when, under one reading,
more than half of the twelve crystals qualify, including 12040g1 and 14053g2. A crystal with one NRM
measurement has no k and no test, and counts as not qualifying.

In [ ]:
rows = []
for i, name in enumerate(T1):
    r = {"crystal": name}
    for step, lab in (("0", "nrm"), ("590", "590.0"), ("590.3", "590.3"), ("400", "400.0"), ("400.3", "400.3")):
        X = grp(name, step)
        if len(X) == 0:
            continue
        f = A.fisher(X)
        r[f"{lab} n"] = f["n"]
        r[f"{lab} k"] = round(f["k"], 1) if f["k"] == f["k"] else np.nan
        r[f"{lab} a95"] = round(f["a95"], 0) if f["a95"] == f["a95"] else np.nan
        r[f"{lab} p"] = A.watson_p(f["n"], f["R"], seed=4000 + 10 * i + len(r))
        r[f"{lab} mean moment"] = float(np.linalg.norm(X, axis=1).mean())
        r[f"{lab} dec,inc"] = (round(f["dec"], 1), round(f["inc"], 1))
    rows.append(r)
cr = pd.DataFrame(rows).set_index("crystal")
cols = [c for c in cr if c.split()[0] in ("nrm", "590.0", "590.3")]
print(cr[cols].to_string())

knrm = cr["nrm k"]
nrm_a = knrm > cr["590.0 k"]
nrm_b = cr["nrm p"] < 0.05
c590_a = cr["590.0 k"] < knrm
c590_b = cr["590.0 p"] >= 0.05
c5903_a = cr["590.3 k"] < knrm
c5903_b = cr["590.3 p"] >= 0.05
summ = pd.DataFrame({"NRM consistent (a)": nrm_a, "NRM consistent (b)": nrm_b, "590.0 inconsistent (a)": c590_a,
                     "590.0 inconsistent (b)": c590_b, "590.3 inconsistent (a)": c5903_a, "590.3 inconsistent (b)": c5903_b})
print(summ.to_string())
print("Counts of 12:", summ.sum().astype(int).to_dict())
print("12040g1 and 14053g2:", summ.loc[["12040g1", "14053g2"]].to_dict("index"))
print(f"590.0 mean moments: {cr['590.0 mean moment'].min():.3g} to {cr['590.0 mean moment'].max():.3g} emu; "
      f"above the 8e-11 emu threshold: {int((cr['590.0 mean moment'] > A.THRESHOLD_EMU).sum())}/12; "
      f"single 590.0 measurements above it: {sum(int((np.linalg.norm(grp(n, '590'), axis=1) > A.THRESHOLD_EMU).sum()) for n in T1)}"
      f"/{sum(len(grp(n, '590')) for n in T1)}")
print(f"590.3 mean moments: {cr['590.3 mean moment'].min():.3g} to {cr['590.3 mean moment'].max():.3g} emu; "
      f"NRM mean moments: {cr['nrm mean moment'].min():.3g} to {cr['nrm mean moment'].max():.3g} emu")

**Result: fig3-nrm-consistent reproduced.** NRM replicates are more tightly grouped than the 590.0
replicates in 10 of 12 crystals (reading a; 8 of 12 under reading b), including 12040g1 and 14053g2;
71055g1 and 71055g3 have one NRM measurement each.

**Result: fig3-590-inconsistent reproduced.** Watson's test does not reject randomness of the 590.0
replicates in any of the 12 crystals (reading b), and their k is below the NRM's in the 10 crystals
with two or more NRM measurements (reading a). The 590.0 moments, 1.2e-10 to 5.7e-10 emu on average,
are all above the 8e-11 emu threshold.

**Result: fig3-alteration-check-inconsistent reproduced.** The same holds for the 590.3 replicates: 12
of 12 under reading (b), 10 of 12 under reading (a).

## crystal-14053g1-400c-stable

> Crystal 14053g1, measured after heating to 400 C during initial tests, held a stable remanence,
> which was removed by treatment at 590 C.

Criterion: applies if the deposit holds 400 C measurements for 14053g1. (a) the 400 C replicates are
consistent under fig3-nrm-consistent's readings and the 590.0 replicates inconsistent; or (b) the 400
C direction lies within the alpha95 of the Fisher mean of the NRM replicates. Reproduced when the
590.0 replicates are inconsistent and (a) or (b) holds.

In [ ]:
r = cr.loc["14053g1"]
print("14053g1 rows at 400 C:")
print(crystals["14053g1"]["data"].query("step.str.startswith('400')").to_string(index=False))
print(f"NRM: n {r['nrm n']}, mean {r['nrm dec,inc']}, k {r['nrm k']}, a95 {r['nrm a95']}, mean moment {r['nrm mean moment']:.3g} emu")
print(f"400.0: n {r['400.0 n']}, mean {r['400.0 dec,inc']}, k {r['400.0 k']}, a95 {r['400.0 a95']}, Watson p {r['400.0 p']:.4f}, mean moment {r['400.0 mean moment']:.3g}")
print(f"400.3: n {r['400.3 n']}, mean {r['400.3 dec,inc']}, k {r['400.3 k']}")
print(f"590.0: n {r['590.0 n']}, k {r['590.0 k']}, Watson p {r['590.0 p']:.3f}, mean moment {r['590.0 mean moment']:.3g}")
a_k = r["400.0 k"] > r["590.0 k"]
a_w = r["400.0 p"] < 0.05
inc590 = (r["590.0 k"] < r["nrm k"]) or (r["590.0 p"] >= 0.05)
m400 = grp("14053g1", "400").sum(axis=0)
mnrm = grp("14053g1", "0")
mnrm = (mnrm / np.linalg.norm(mnrm, axis=1, keepdims=True)).sum(axis=0)
ang = A.angle(m400, mnrm)
print(f"(a) 400 C consistent: k above 590.0's {a_k}, Watson rejects {a_w}; 590.0 inconsistent {inc590}")
print(f"(b) angle between the 400 C mean and the NRM mean {ang:.1f} deg, NRM alpha95 {r['nrm a95']} deg: within {ang <= r['nrm a95']}")
for i, x in enumerate(grp("14053g1", "400")):
    print(f"    400 C replicate {i + 1}: {A.angle(x, mnrm):.1f} deg from the NRM mean")

**Result: reproduced.** The deposit holds three 400 C measurements of 14053g1 (with a 400.1 and two
400.3 rows). They are grouped (k above that of the 590.0 replicates, and Watson's test rejects
randomness), while the 590.0 replicates are not, so reading (a) holds. Under reading (b) the 400 C
mean lies outside the NRM's alpha95 (numbers above).

## no-alteration-recording-efficiency

> Alteration during the laser heating of the single crystals was not significant; apparent recording
> efficiencies far above 100% were not observed and the alteration checks did not fail.

Criterion: (a) the 590.3 moment does not exceed the 590.0 moment by more than 8e-11 emu, and the
590.3 directions are inconsistent; (b) the 590.4 TRM (|590.4 - 590.3|) is no more than about twice the
590.1 TRM (|590.1 - 590.0|); a ratio of the 590.4 TRM to twice the 590.1 TRM above 2 fails (a choice
not from the paper). Reproduced when every crystal passes under one reading.

In [ ]:
rows = []
for name in T1:
    on, off, alt, on40 = grp(name, "590.1")[0], grp(name, "590"), grp(name, "590.3"), grp(name, "590.4")[0]
    for rep in ("i", "ii"):
        o = off[0] if rep == "i" else off.mean(axis=0)
        al = alt[0] if rep == "i" else alt.mean(axis=0)
        m_off = np.linalg.norm(off[0]) if rep == "i" else np.linalg.norm(off, axis=1).mean()
        m_alt = np.linalg.norm(alt[0]) if rep == "i" else np.linalg.norm(alt, axis=1).mean()
        ratio = np.linalg.norm(on40 - al) / (2 * np.linalg.norm(on - o))
        rows.append({"crystal": name, "replicates": rep, "590.3 - 590.0 moment (1e-11 emu)": round((m_alt - m_off) * 1e11, 1),
                     "a_moment_ok": m_alt - m_off <= A.THRESHOLD_EMU, "a_dirs_inconsistent": bool(summ.loc[name, "590.3 inconsistent (b)"]),
                     "TRM40/(2 TRM20)": round(ratio, 3), "b_ok": ratio <= 2})
alt = pd.DataFrame(rows)
alt["a_ok"] = alt.a_moment_ok & alt.a_dirs_inconsistent
print(alt.to_string(index=False))
for rep in ("i", "ii"):
    x = alt[alt.replicates == rep]
    print(f"replicates {rep}: pass (a) {int(x.a_ok.sum())}/12, pass (b) {int(x.b_ok.sum())}/12; "
          f"TRM40/(2 TRM20) {x['TRM40/(2 TRM20)'].min():.2f} to {x['TRM40/(2 TRM20)'].max():.2f}")

**Result: reproduced.** Under reading (b) every crystal passes: the TRM acquired in 40 uT is 0.77 to
1.17 times twice the TRM acquired in 20 uT, close to linear recording and far from the cutoff of 2.
Under reading (a) 11 of 12 pass with first replicates and 10 of 12 with means; 14053g1's 590.3
moment exceeds its 590.0 moment by 9.8e-11 to 1.21e-10 emu, above the 8e-11 emu allowance.

## interpretation-impact-no-dynamo

> Together with ref. 1, the data are consistent with impact plasma magnetization of the 64455 glass
> and with the absence of a long-lived lunar core dynamo recorded in the 3.9 to 3.2 Ga crystals.

The claims file says this interpretation is not tested; it rests on ref. 1 and on models of impact
plasmas. It is recorded as blocked. The outcomes of the four claims it names are: fig2-changing-field
reproduced, glass-directions-not-random partially reproduced, fig3-590-inconsistent reproduced,
table1-max-intensity reproduced.

## Summary

| Claim | Outcome |
|---|---|
| glass-specimen-set (link) | reproduced |
| glass-success-rate | reproduced |
| table2-fit-statistics | partially reproduced (assumes averaged replicates) |
| table2-no-fit | reproduced |
| fig1-linear-components | reproduced (assumes averaged replicates) |
| fig2-changing-field | reproduced (assumes averaged replicates) |
| glass-directions-not-random | partially reproduced |
| glass-af-no-change | reproduced |
| table1-max-intensity | reproduced |
| baseline-range | reproduced |
| example-14053g1-threshold | reproduced under the misprint reading |
| fig3-nrm-consistent | reproduced |
| fig3-590-inconsistent | reproduced |
| fig3-alteration-check-inconsistent | reproduced |
| crystal-14053g1-400c-stable | reproduced |
| no-alteration-recording-efficiency | reproduced |
| interpretation-impact-no-dynamo | blocked (interpretation, not tested) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: the paper used PmagPy and gives no formulas; MAD, R2 and
pTRM checks are computed here by the claims file's readings, and repeated measurements are averaged,
an assumption that reproduces most of Table 2. No test of randomness is named by the paper; Watson's
test is the claims file's choice.

What was found about the dataset:

- The "replicates" of four glass sheets (g2, ss8, ss39, ss48) are one measurement multiplied by
  factors near 1 (0.95 to 1.03), applied to moment, declination and inclination; they are not
  repeated measurements.
- One inclination of -136 deg (ss19, 10 mT), one declination of 361.7 deg (ss8), an empty row (ss18),
  and an extra column in ss48.
- The number of repeated measurements varies from 2 to 19 per step, ss6's 250.1 step holds two groups
  of directions, and the deposit does not say how repeated measurements were combined.
- The glass laboratory field varies between sheets (14.4, 15 and 18 uT); the paper states only 18 uT.
- The Technical Validation example for 14053g1 uses the numbers of 12040g1.
- The crystals' 590 C zero-field moments are all above the 8e-11 emu threshold the paper uses.